# 10. Feature Engineering (피처 엔지니어링)

## 목적
전처리된 데이터에서 ML 학습용 파생 피처 생성

## 입력
- `preprocessed.csv`: 전처리된 데이터 (결측 처리 완료)
- `sepsis_cohort.csv`: SOFA 점수 등 코호트 정보

## 출력
- `features_final.csv`: 최종 피처셋 (피처 only)
- `labels_final.csv`: ID + 레이블

## 최종 피처 구성
| 구분 | 피처 | 개수 |
|------|------|------|
| Vitals | hr, rr, spo2, sbp, dbp, mbp, temp | 7 |
| Vitals 통계 | hr_max, rr_max, spo2_min, sbp_min | 4 |
| Labs | creatinine, wbc, platelets, potassium, sodium, lactate, ph, bilirubin | 8 |
| GCS | gcs_eye, gcs_verbal, gcs_motor, gcs_total | 4 |
| Urine | urine_ml_6h, urine_ml_kg_hr_avg, oliguria_flag | 3 |
| 파생 (hemodynamic) | shock_index, pulse_pressure, map_below_65 | 3 |
| 파생 (scoring) | news_score, mews_score | 2 |
| 정적 (cohort) | anchor_age, gender, sofa_total, first_careunit_* | 4+ |
| Flags | lactate_missing, abga_checked, bilirubin_missing, urine_missing | 4 |
| ~~Delta~~ | ~~hr_delta, sbp_delta, ... (9개)~~ | **제거** |
| Slope | hr_slope, sbp_slope, spo2_slope, lactate_slope, gcs_total_slope, rr_slope, temp_slope, creatinine_slope, ph_slope | 9 |

## 피처 엔지니어링 범위
- [x] Hemodynamic 파생 (Shock Index, Pulse Pressure, MAP<65)
- [x] 중증도 점수 (NEWS, MEWS) - 벡터화
- [x] 정적 피처 (SOFA, gender, careunit)
- [ ] ~~Delta 피처 9개~~ → **제거** (slope와 임상적 의미 중복, 정보 손실 없음)
- [x] Slope 피처 (9개)
- [x] 인구통계학적 피처 (anchor_age)

In [1]:
from pathlib import Path
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
import pandas as pd
import numpy as np
import os

INPUT_DIR = PROCESSED_DIR
OUTPUT_DIR = PROCESSED_DIR

print("=== 10. Feature Engineering 시작 ===")

=== 10. Feature Engineering 시작 ===


## Step 1: 데이터 로드

In [2]:
print("Step 1: 데이터 로드")

df = pd.read_csv(INPUT_DIR / "preprocessed.csv")
df = df.sort_values(['stay_id', 'observation_hour']).reset_index(drop=True)

print(f"✓ preprocessed.csv 로드 완료: {len(df):,} rows")
print(f"✓ 고유 환자: {df['stay_id'].nunique():,}명")

# 코호트에서 SOFA, gender, careunit 가져오기
df_cohort = pd.read_csv(INPUT_DIR / "sepsis_cohort.csv")
cohort_cols = ['stay_id', 'sofa_total', 'sofa_resp', 'sofa_coag', 'sofa_liver',
               'sofa_cardio', 'sofa_cns', 'sofa_renal', 'gender', 'first_careunit']
# 실제 존재하는 컬럼만
cohort_cols = [c for c in cohort_cols if c in df_cohort.columns]
print(f"✓ sepsis_cohort.csv 로드: 코호트 컬럼 = {cohort_cols}")

# 이미 있는 컬럼은 제외하고 병합
merge_cols = [c for c in cohort_cols if c not in df.columns or c == 'stay_id']
if len(merge_cols) > 1:  # stay_id 외 추가 컬럼이 있으면
    df = df.merge(df_cohort[merge_cols], on='stay_id', how='left')
    print(f"✓ 코호트 정보 병합 완료: {[c for c in merge_cols if c != 'stay_id']}")

print(f"\n컬럼 수: {len(df.columns)}개")

Step 1: 데이터 로드
✓ preprocessed.csv 로드 완료: 158,985 rows
✓ 고유 환자: 4,713명
✓ sepsis_cohort.csv 로드: 코호트 컬럼 = ['stay_id', 'sofa_total', 'sofa_resp', 'sofa_coag', 'sofa_liver', 'sofa_cardio', 'sofa_cns', 'sofa_renal', 'gender', 'first_careunit']
✓ 코호트 정보 병합 완료: ['sofa_total', 'sofa_resp', 'sofa_coag', 'sofa_liver', 'sofa_cardio', 'sofa_cns', 'sofa_renal']

컬럼 수: 64개


In [3]:
# 피처 그룹 정의 (전처리에서 넘어온 것)
vital_cols = ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp', 'temp']
vital_stat_cols = ['hr_max', 'rr_max', 'spo2_min', 'sbp_min']
lab_cols = ['creatinine', 'wbc', 'platelets', 'potassium', 'sodium', 'lactate', 'ph', 'bilirubin']
gcs_cols = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']
urine_cols = ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']

# 전처리에서 생성된 플래그
flag_cols = ['lactate_missing', 'abga_checked', 'bilirubin_missing', 'urine_missing']

# 현재 존재하는 피처 확인
print("\n=== 입력 피처 확인 ===")
for name, cols in [('Vitals', vital_cols), ('Vital Stats', vital_stat_cols), 
                   ('Labs', lab_cols), ('GCS', gcs_cols), ('Urine', urine_cols),
                   ('Flags', flag_cols)]:
    existing = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    print(f"  {name}: {len(existing)}/{len(cols)} 존재")
    if missing:
        print(f"    ⚠️ 없음: {missing}")

# 코호트 정보 확인
print("\n[Cohort 정보]")
for col in ['sofa_total', 'gender', 'first_careunit', 'anchor_age']:
    if col in df.columns:
        print(f"  ✓ {col}: {df[col].nunique()} unique")
    else:
        print(f"  ✗ {col}: 없음")


=== 입력 피처 확인 ===
  Vitals: 7/7 존재
  Vital Stats: 4/4 존재
  Labs: 8/8 존재
  GCS: 4/4 존재
  Urine: 3/3 존재
  Flags: 4/4 존재

[Cohort 정보]
  ✓ sofa_total: 22 unique
  ✓ gender: 2 unique
  ✓ first_careunit: 13 unique
  ✓ anchor_age: 73 unique


## Step 2: Hemodynamic 파생 피처

**Shock Index = HR / SBP**
- 정상: 0.5 ~ 0.7, >1.0: 쇼크 의심

**Pulse Pressure = SBP - DBP**
- 감소(< 25) = 심박출량 저하 → 쇼크 지표

**MAP < 65 Flag**
- Sepsis-3 정의: vasopressor 필요한 MAP < 65 → 패혈증 쇼크 기준

In [4]:
print("Step 2: Hemodynamic 파생 피처 생성")

# --- 2-1: Shock Index = HR / SBP ---
df['shock_index'] = df['hr'] / df['sbp'].replace(0, np.nan)
df['shock_index'] = df['shock_index'].fillna(df['shock_index'].median())
df['shock_index'] = df['shock_index'].clip(0, 3)

print(f"✓ shock_index: mean={df['shock_index'].mean():.3f}, median={df['shock_index'].median():.3f}")
print(f"  range: {df['shock_index'].min():.3f} ~ {df['shock_index'].max():.3f}")

# --- 2-2: Pulse Pressure = SBP - DBP ---
df['pulse_pressure'] = df['sbp'] - df['dbp']
df['pulse_pressure'] = df['pulse_pressure'].clip(0, 200)  # 음수 방지 + 이상치 클리핑

print(f"✓ pulse_pressure: mean={df['pulse_pressure'].mean():.1f}, median={df['pulse_pressure'].median():.1f}")

# --- 2-3: MAP < 65 Flag (Sepsis-3 저혈압 기준) ---
df['map_below_65'] = (df['mbp'] < 65).astype(int)

map_low_count = df['map_below_65'].sum()
print(f"✓ map_below_65: {map_low_count:,}건 ({map_low_count/len(df)*100:.1f}%)")

Step 2: Hemodynamic 파생 피처 생성
✓ shock_index: mean=0.829, median=0.795
  range: 0.195 ~ 3.000
✓ pulse_pressure: mean=52.4, median=51.0
✓ map_below_65: 53,519건 (33.7%)


## Step 3: 중증도 점수 (MEWS, NEWS) - 벡터화

### MEWS (Modified Early Warning Score)
- HR, SBP, RR, Temp 기반
- 범위: 0~14

### NEWS (National Early Warning Score)
- SpO2, SBP, HR, RR, Temp, GCS 기반
- 범위: 0~20

**벡터화**: `apply(axis=1)` 대신 `np.where` 사용 → 5~15분 → 2~5초

In [5]:
print("Step 3: 중증도 점수 계산 (벡터화)")

# --- MEWS 벡터화 ---
def calc_mews_vectorized(df):
    """
    Modified Early Warning Score (벡터화)
    - HR: 40미만/130초과(3), 50미만/110초과(2), 60미만/100초과(1)
    - SBP: 70미만(3), 80미만(2), 100미만(1)
    - RR: 9미만/30초과(3), 25초과(2), 12미만/20초과(1)
    - Temp: 35미만/39초과(2), 36미만/38초과(1)
    """
    score = pd.Series(0, index=df.index)
    
    # HR
    hr = df['hr']
    score += np.where((hr < 40) | (hr > 130), 3,
             np.where((hr < 50) | (hr > 110), 2,
             np.where((hr < 60) | (hr > 100), 1, 0)))
    
    # SBP
    sbp = df['sbp']
    score += np.where(sbp < 70, 3,
             np.where(sbp < 80, 2,
             np.where(sbp < 100, 1, 0)))
    
    # RR
    rr = df['rr']
    score += np.where((rr < 9) | (rr > 30), 3,
             np.where(rr > 25, 2,
             np.where((rr < 12) | (rr > 20), 1, 0)))
    
    # Temp
    temp = df['temp']
    score += np.where((temp < 35) | (temp > 39), 2,
             np.where((temp < 36) | (temp > 38), 1, 0))
    
    return score

df['mews_score'] = calc_mews_vectorized(df)
print(f"✓ mews_score: mean={df['mews_score'].mean():.2f}, max={df['mews_score'].max()}")

Step 3: 중증도 점수 계산 (벡터화)
✓ mews_score: mean=1.49, max=10


In [6]:
# --- NEWS 벡터화 ---
def calc_news_vectorized(df):
    """
    National Early Warning Score (벡터화)
    - RR: 8이하/25이상(3), 21-24(2), 9-11(1)
    - SpO2: 91이하(3), 92-93(2), 94-95(1)
    - Temp: 35이하/39.1이상(3), 35.1-36/38.1-39(1)
    - SBP: 90이하/220이상(3), 91-100(2), 101-110(1)
    - HR: 40이하/131이상(3), 111-130(2), 41-50/91-110(1)
    - Consciousness: GCS<15 → +3
    """
    score = pd.Series(0, index=df.index)
    
    # RR
    rr = df['rr']
    score += np.where((rr <= 8) | (rr >= 25), 3,
             np.where((rr >= 21) & (rr <= 24), 2,
             np.where((rr >= 9) & (rr <= 11), 1, 0)))
    
    # SpO2
    spo2 = df['spo2']
    score += np.where(spo2 <= 91, 3,
             np.where(spo2 <= 93, 2,
             np.where(spo2 <= 95, 1, 0)))
    
    # Temp
    temp = df['temp']
    score += np.where((temp <= 35) | (temp >= 39.1), 3,
             np.where(((temp > 35) & (temp <= 36)) | ((temp >= 38.1) & (temp < 39.1)), 1, 0))
    
    # SBP
    sbp = df['sbp']
    score += np.where((sbp <= 90) | (sbp >= 220), 3,
             np.where((sbp >= 91) & (sbp <= 100), 2,
             np.where((sbp >= 101) & (sbp <= 110), 1, 0)))
    
    # HR
    hr = df['hr']
    score += np.where((hr <= 40) | (hr >= 131), 3,
             np.where((hr >= 111) & (hr <= 130), 2,
             np.where(((hr >= 41) & (hr <= 50)) | ((hr >= 91) & (hr <= 110)), 1, 0)))
    
    # Consciousness (GCS 기반)
    gcs = df['gcs_total']
    score += np.where(gcs < 15, 3, 0)
    
    return score

df['news_score'] = calc_news_vectorized(df)
print(f"✓ news_score: mean={df['news_score'].mean():.2f}, max={df['news_score'].max()}")

✓ news_score: mean=5.14, max=18


## Step 4: Delta 피처 — 제거

> **설계 결정**: delta 피처 9개(`hr_delta`, `sbp_delta`, `spo2_delta`, `lactate_delta`,
> `gcs_total_delta`, `rr_delta`, `temp_delta`, `creatinine_delta`, `ph_delta`)는 제거합니다.
>
> **이유**: Step 5의 slope 피처가 최근 N-윈도우 선형 추세를 포착하여 delta(단순 전후 차이)보다
> 더 많은 정보를 담고 있음. 두 피처를 동시에 포함하면 임상적 의미가 중복되어 모델 해석성 저하.
>
> **유지**: Slope 피처 9개 (`hr_slope`, `sbp_slope`, ... `ph_slope`) → Step 5에서 계산

In [7]:
print("Step 4: Delta 피처 — 제거 (설계 결정)")
print()
print("  delta 피처 9개는 slope 피처(Step 5)와 임상적 의미가 중복되어 제거합니다.")
print("  - 제거: hr_delta, sbp_delta, spo2_delta, rr_delta, temp_delta,")
print("           lactate_delta, creatinine_delta, ph_delta, gcs_total_delta")
print()
print("  유지: slope 피처 9개 → Step 5에서 계산")

# delta 컬럼이 이미 df에 있을 경우 제거 (재실행 안전성)
delta_cols = [f'{c}_delta' for c in ['hr', 'sbp', 'spo2', 'rr', 'temp',
                                      'lactate', 'creatinine', 'ph', 'gcs_total']]
existing_delta = [c for c in delta_cols if c in df.columns]
if existing_delta:
    df.drop(columns=existing_delta, inplace=True)
    print(f"\n  ✓ 기존 delta 컬럼 {len(existing_delta)}개 제거 완료")

Step 4: Delta 피처 — 제거 (설계 결정)

  delta 피처 9개는 slope 피처(Step 5)와 임상적 의미가 중복되어 제거합니다.
  - 제거: hr_delta, sbp_delta, spo2_delta, rr_delta, temp_delta,
           lactate_delta, creatinine_delta, ph_delta, gcs_total_delta

  유지: slope 피처 9개 → Step 5에서 계산


## Step 5: Slope 피처 (추세)

최근 N개 윈도우(기본 6h)의 **선형 기울기(OLS slope)** 로 추세를 반영

- **단위**: value / hour
- **측정값 2개 미만**: 0으로 설정

In [8]:
print("Step 5: Slope 피처 생성")

# 최근 N개 윈도우 기준 선형 기울기
SLOPE_WINDOW = 6  # 6시간(=6개 윈도우)

slope_features = ['hr', 'sbp', 'spo2', 'rr', 'temp', 'lactate', 'creatinine', 'ph', 'gcs_total']

def rolling_slope(t: pd.Series, y: pd.Series, window: int) -> pd.Series:
    t = t.astype(float)
    y = y.astype(float)

    # y가 없는 지점은 제외
    mask = t.notna() & y.notna()
    t = t.where(mask)
    y = y.where(mask)

    n = y.rolling(window=window, min_periods=2).count()
    sum_t = t.rolling(window=window, min_periods=2).sum()
    sum_y = y.rolling(window=window, min_periods=2).sum()
    sum_t2 = (t ** 2).rolling(window=window, min_periods=2).sum()
    sum_ty = (t * y).rolling(window=window, min_periods=2).sum()

    denom = n * sum_t2 - (sum_t ** 2)
    slope = (n * sum_ty - sum_t * sum_y) / denom
    slope = slope.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return slope

for col in slope_features:
    if col not in df.columns:
        print(f"  ⚠️ {col} 없음, 스킵")
        continue

    slope_col = f"{col}_slope"

    df[slope_col] = (
        df.groupby('stay_id', group_keys=False)
          .apply(lambda g: rolling_slope(g['observation_hour'], g[col], SLOPE_WINDOW))
    )

    print(f"  ✓ {slope_col}: mean={df[slope_col].mean():.4f}, std={df[slope_col].std():.4f}")

print(f"\n✓ Slope 피처 {len(slope_features)}개 생성 완료 (window={SLOPE_WINDOW})")

Step 5: Slope 피처 생성
  ✓ hr_slope: mean=-0.0985, std=1.7528
  ✓ sbp_slope: mean=0.1169, std=2.8962
  ✓ spo2_slope: mean=-0.0039, std=0.8070
  ✓ rr_slope: mean=-0.0039, std=0.6294
  ✓ temp_slope: mean=-0.0013, std=0.1171
  ✓ lactate_slope: mean=-0.0007, std=0.1011
  ✓ creatinine_slope: mean=0.0016, std=0.0974
  ✓ ph_slope: mean=-0.0000, std=0.0055
  ✓ gcs_total_slope: mean=0.0118, std=0.3180

✓ Slope 피처 9개 생성 완료 (window=6)


## Step 6: 정적 피처 (Cohort 기반)

- **anchor_age**: 나이 (08에서 병합됨)
- **gender**: 성별 → binary encoding (M=1, F=0)
- **sofa_total**: 입실 시 SOFA 점수 (sepsis 사망률 최강 예측인자)
- **first_careunit**: ICU 유형 → binary encoding (상위 유닛만)

In [9]:
print("Step 6: 정적 피처 처리")

# --- 6-1: anchor_age ---
if 'anchor_age' in df.columns:
    print(f"✓ anchor_age: mean={df['anchor_age'].mean():.1f}, range={df['anchor_age'].min()}~{df['anchor_age'].max()}")
else:
    print("⚠️ anchor_age 없음")

# --- 6-2: gender → binary ---
if 'gender' in df.columns:
    df['gender'] = df['gender'].map({'M': 1, 'F': 0}).fillna(0).astype(int)
    male_pct = df['gender'].mean() * 100
    print(f"✓ gender: M(1)={male_pct:.1f}%, F(0)={100-male_pct:.1f}%")
else:
    print("⚠️ gender 없음")

# --- 6-3: sofa_total ---
if 'sofa_total' in df.columns:
    df['sofa_total'] = df['sofa_total'].fillna(df['sofa_total'].median())
    print(f"✓ sofa_total: mean={df['sofa_total'].mean():.1f}, median={df['sofa_total'].median():.0f}")
else:
    print("⚠️ sofa_total 없음 — sepsis_cohort.csv에서 가져오기 필요")

# --- 6-4: first_careunit → binary features ---
if 'first_careunit' in df.columns:
    print(f"\n  ICU 유형 분포:")
    icu_counts = df['first_careunit'].value_counts()
    for icu, cnt in icu_counts.items():
        print(f"    {icu}: {cnt:,} ({cnt/len(df)*100:.1f}%)")
    
    # 상위 ICU 유형을 binary feature로 생성
    # Medical ICU가 가장 흔하므로 기준(reference)으로 두고, 나머지를 flag화
    icu_types = ['MICU', 'SICU', 'CCU', 'MICU/SICU', 'TSICU']
    for icu in icu_types:
        col_name = f'icu_{icu.lower().replace("/", "_")}'
        df[col_name] = df['first_careunit'].str.contains(icu, case=False, na=False).astype(int)
        if df[col_name].sum() > 0:
            print(f"  ✓ {col_name}: {df[col_name].sum():,}건")
    
    # 원본 문자열 컬럼은 최종 DataFrame 구성 시 제외
else:
    print("⚠️ first_careunit 없음")

Step 6: 정적 피처 처리
✓ anchor_age: mean=65.6, range=18~91
✓ gender: M(1)=55.8%, F(0)=44.2%
✓ sofa_total: mean=5.9, median=5

  ICU 유형 분포:
    Medical/Surgical Intensive Care Unit (MICU/SICU): 43,975 (27.7%)
    Medical Intensive Care Unit (MICU): 32,673 (20.6%)
    Surgical Intensive Care Unit (SICU): 29,506 (18.6%)
    Neuro Intermediate: 13,235 (8.3%)
    Trauma SICU (TSICU): 12,905 (8.1%)
    Coronary Care Unit (CCU): 11,399 (7.2%)
    Cardiac Vascular Intensive Care Unit (CVICU): 7,943 (5.0%)
    Neuro Stepdown: 3,917 (2.5%)
    Neuro Surgical Intensive Care Unit (Neuro SICU): 2,941 (1.8%)
    PACU: 210 (0.1%)
    Surgery/Vascular/Intermediate: 183 (0.1%)
    Intensive Care Unit (ICU): 86 (0.1%)
    Neurology: 12 (0.0%)
  ✓ icu_micu: 76,648건
  ✓ icu_sicu: 89,327건
  ✓ icu_ccu: 11,399건
  ✓ icu_micu_sicu: 43,975건
  ✓ icu_tsicu: 12,905건


## Step 7: 최종 피처 정리 및 검증

In [10]:
print("Step 7: 최종 피처 정리")

# ICU binary 컬럼 자동 탐지
icu_binary_cols = [c for c in df.columns if c.startswith('icu_') and c != 'icu_mortality']

# 최종 피처 정의 (delta 피처 제거, slope 유지)
final_features = {
    'vitals': ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp', 'temp'],
    'vitals_stat': ['hr_max', 'rr_max', 'spo2_min', 'sbp_min'],
    'labs': ['creatinine', 'wbc', 'platelets', 'potassium', 'sodium', 'lactate', 'ph', 'bilirubin'],
    'gcs': ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total'],
    'urine': ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag'],
    'derived_hemodynamic': ['shock_index', 'pulse_pressure', 'map_below_65'],
    'derived_scoring': ['news_score', 'mews_score'],
    'static': ['anchor_age', 'gender', 'sofa_total', 'observation_hour'] + icu_binary_cols,
    'flags': ['lactate_missing', 'abga_checked', 'bilirubin_missing', 'urine_missing'],
    # delta 피처 제거 — slope 피처로 대체
    'slope': [f'{c}_slope' for c in ['hr', 'sbp', 'spo2', 'rr', 'temp',
                                      'lactate', 'creatinine', 'ph', 'gcs_total']],
}

# 전체 피처 리스트
all_features = []
for group, cols in final_features.items():
    all_features.extend(cols)

print(f"\n=== 최종 피처 구성 ({len(all_features)}개) ===")
for group, cols in final_features.items():
    existing = [c for c in cols if c in df.columns]
    missing_cols = [c for c in cols if c not in df.columns]
    status = "✓" if len(missing_cols) == 0 else "⚠️"
    print(f"  {status} [{group}] {len(existing)}/{len(cols)}개")
    if missing_cols:
        print(f"      없음: {missing_cols}")

print(f"\n  → delta 피처 9개 제거됨 (설계 결정)")

Step 7: 최종 피처 정리

=== 최종 피처 구성 (53개) ===
  ✓ [vitals] 7/7개
  ✓ [vitals_stat] 4/4개
  ✓ [labs] 8/8개
  ✓ [gcs] 4/4개
  ✓ [urine] 3/3개
  ✓ [derived_hemodynamic] 3/3개
  ✓ [derived_scoring] 2/2개
  ✓ [static] 9/9개
  ✓ [flags] 4/4개
  ✓ [slope] 9/9개

  → delta 피처 9개 제거됨 (설계 결정)


In [11]:
# 결측 확인
print("\n=== 결측 확인 ===")
leak_cols = ['icu_mortality']
existing_features = [c for c in all_features if c in df.columns and c not in leak_cols]
total_missing = df[existing_features].isna().sum().sum()

if total_missing == 0:
    print(f"✓ {len(existing_features)}개 피처 모두 결측 없음")
else:
    print(f"⚠️ 총 결측: {total_missing:,}건")
    missing_detail = df[existing_features].isna().sum()
    print(missing_detail[missing_detail > 0])


=== 결측 확인 ===
✓ 53개 피처 모두 결측 없음


In [12]:
# 피처 기술 통계
print("\n=== 피처 기술 통계 ===")
print(df[existing_features].describe().round(2))


=== 피처 기술 통계 ===


              hr         rr       spo2        sbp        dbp        mbp  \
count  158985.00  158985.00  158985.00  158985.00  158985.00  158985.00   
mean       86.96      20.07      93.53     108.45      56.04      70.73   
std        18.09       4.96       3.79      18.73      12.83      13.40   
min        24.00       6.00      50.00      40.00      20.00      30.00   
25%        74.00      16.50      92.00      95.00      47.00      61.00   
50%        86.00      19.50      94.00     107.00      55.00      70.00   
75%        99.00      23.00      96.00     120.00      64.00      79.00   
max       181.00      60.00     100.00     224.00     147.00     153.00   

            temp     hr_max     rr_max   spo2_min  ...  urine_missing  \
count  158985.00  158985.00  158985.00  158985.00  ...      158985.00   
mean       36.87      95.43      24.89      93.53  ...           0.18   
std         0.52      19.59       6.38       3.79  ...           0.38   
min        31.89      28.00     

## Step 7: 최종 DataFrame 구성

In [13]:
print("Step 8: 최종 DataFrame 구성")

# ID 컬럼
id_cols = ['stay_id', 'subject_id', 'hadm_id', 'observation_hour', 'observation_start', 'observation_end']

# 레이블 컬럼
label_cols = [col for col in df.columns if 'next_' in col]

# leakage 컬럼 (미래 정보)
leak_cols = ['icu_mortality']

# 최종 피처 컬럼 (leakage 제외)
feature_cols_final = [c for c in all_features if c in df.columns and c not in leak_cols]

# 피처 / 레이블 분리
df_features = df[feature_cols_final].copy()
df_labels = df[[c for c in id_cols if c in df.columns] + label_cols].copy()

print(f"\n=== 최종 DataFrame ===")
print(f"  행 수: {len(df_features):,}")
print(f"  피처 컬럼 수: {len(df_features.columns)}")
print(f"  레이블 컬럼 수: {len(label_cols)}")
print(f"  ID 컬럼 수: {len([c for c in id_cols if c in df.columns])}")


Step 8: 최종 DataFrame 구성

=== 최종 DataFrame ===
  행 수: 158,985
  피처 컬럼 수: 53
  레이블 컬럼 수: 12
  ID 컬럼 수: 6


## Step 8: 저장

In [14]:
print("Step 9: 저장")

features_path = OUTPUT_DIR / "features_final.csv"
labels_path   = OUTPUT_DIR / "labels_final.csv"

df_features.to_csv(features_path, index=False)
df_labels.to_csv(labels_path, index=False)

features_size = os.path.getsize(features_path) / (1024 * 1024)
labels_size   = os.path.getsize(labels_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: features_final.csv")
print(f"  - 파일 크기: {features_size:.2f} MB")
print(f"  - 행 수: {len(df_features):,}개")
print(f"  - 컬럼 수: {len(df_features.columns)}개")
print(f"  - 경로: {features_path}")

print(f"\n✓ 저장 완료: labels_final.csv")
print(f"  - 파일 크기: {labels_size:.2f} MB")
print(f"  - 행 수: {len(df_labels):,}개")
print(f"  - 컬럼 수: {len(df_labels.columns)}개")
print(f"  - 경로: {labels_path}")


Step 9: 저장

✓ 저장 완료: features_final.csv
  - 파일 크기: 49.81 MB
  - 행 수: 158,985개
  - 컬럼 수: 53개
  - 경로: DATA/processed/features_final.csv

✓ 저장 완료: labels_final.csv
  - 파일 크기: 14.24 MB
  - 행 수: 158,985개
  - 컬럼 수: 18개
  - 경로: DATA/processed/labels_final.csv


In [15]:
# 피처 목록 출력
print(f"\n=== 최종 피처 목록 ({len(existing_features)}개) ===")
for group, cols in final_features.items():
    print(f"\n[{group.upper()}]")
    for col in cols:
        if col in df_features.columns:
            print(f"  ✓ {col}")
        else:
            print(f"  ✗ {col} (없음)")


=== 최종 피처 목록 (53개) ===

[VITALS]
  ✓ hr
  ✓ rr
  ✓ spo2
  ✓ sbp
  ✓ dbp
  ✓ mbp
  ✓ temp

[VITALS_STAT]
  ✓ hr_max
  ✓ rr_max
  ✓ spo2_min
  ✓ sbp_min

[LABS]
  ✓ creatinine
  ✓ wbc
  ✓ platelets
  ✓ potassium
  ✓ sodium
  ✓ lactate
  ✓ ph
  ✓ bilirubin

[GCS]
  ✓ gcs_eye
  ✓ gcs_verbal
  ✓ gcs_motor
  ✓ gcs_total

[URINE]
  ✓ urine_ml_6h
  ✓ urine_ml_kg_hr_avg
  ✓ oliguria_flag

[DERIVED_HEMODYNAMIC]
  ✓ shock_index
  ✓ pulse_pressure
  ✓ map_below_65

[DERIVED_SCORING]
  ✓ news_score
  ✓ mews_score

[STATIC]
  ✓ anchor_age
  ✓ gender
  ✓ sofa_total
  ✓ observation_hour
  ✓ icu_micu
  ✓ icu_sicu
  ✓ icu_ccu
  ✓ icu_micu_sicu
  ✓ icu_tsicu

[FLAGS]
  ✓ lactate_missing
  ✓ abga_checked
  ✓ bilirubin_missing
  ✓ urine_missing

[SLOPE]
  ✓ hr_slope
  ✓ sbp_slope
  ✓ spo2_slope
  ✓ rr_slope
  ✓ temp_slope
  ✓ lactate_slope
  ✓ creatinine_slope
  ✓ ph_slope
  ✓ gcs_total_slope


In [16]:
# 레이블 분포
print(f"\n=== 레이블 분포 ===")
for col in sorted(label_cols):
    pos_rate = df_labels[col].mean() * 100
    pos_count = df_labels[col].sum()
    print(f"  {col}: {pos_count:,} ({pos_rate:.2f}%)")


=== 레이블 분포 ===
  composite_next_12h: 8,639 (5.43%)
  composite_next_24h: 13,000 (8.18%)
  composite_next_6h: 5,221 (3.28%)
  death_next_12h: 709 (0.45%)
  death_next_24h: 1,765 (1.11%)
  death_next_6h: 319 (0.20%)
  pressor_next_12h: 3,383 (2.13%)
  pressor_next_24h: 5,225 (3.29%)
  pressor_next_6h: 1,989 (1.25%)
  vent_next_12h: 6,418 (4.04%)
  vent_next_24h: 9,472 (5.96%)
  vent_next_6h: 3,847 (2.42%)


In [17]:
print("\n" + "="*50)
print("=== 10. Feature Engineering 완료 ===")
print("="*50)
print(f"\n최종 피처: {len(existing_features)}개")
print(f"샘플 수: {len(df_features):,}개")
print(f"환자 수: {df_labels['stay_id'].nunique():,}명")
print(f"\n다음 단계: 11_modeling.ipynb")


=== 10. Feature Engineering 완료 ===

최종 피처: 53개
샘플 수: 158,985개
환자 수: 4,713명

다음 단계: 11_modeling.ipynb


---
## +a: 검증

In [18]:
# 결측 최종 확인
print("=== 결측 최종 확인 ===")
feature_cols_only = [c for c in all_features if c in df_features.columns]
missing = df_features[feature_cols_only].isna().sum()
if missing.sum() == 0:
    print("✓ 모든 피처 결측 없음")
else:
    print("결측 있는 피처:")
    print(missing[missing > 0])

=== 결측 최종 확인 ===
✓ 모든 피처 결측 없음


In [19]:
# 고상관 피처 확인 (다중공선성)
print("\n=== 고상관 피처 쌍 (>0.90) ===")
feature_cols_only = [c for c in all_features if c in df_features.columns]
corr = df_features[feature_cols_only].corr()

high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.90:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

if high_corr:
    for pair in sorted(high_corr, key=lambda x: -abs(x[2]))[:15]:
        print(f"  {pair[0]} vs {pair[1]}: {pair[2]:.3f}")
    print(f"\n  총 {len(high_corr)}쌍 (tree 모델은 다중공선성 내성이 있으므로 제거 불필요)")
else:
    print("  없음")


=== 고상관 피처 쌍 (>0.90) ===
  spo2 vs spo2_min: 1.000
  sbp vs sbp_min: 1.000
  hr vs hr_max: 0.922

  총 3쌍 (tree 모델은 다중공선성 내성이 있으므로 제거 불필요)


In [20]:
# Flags 분포 확인
print("\n=== Flags 분포 ===")
for col in ['lactate_missing', 'abga_checked', 'bilirubin_missing', 'urine_missing']:
    if col in df_features.columns:
        ones = (df_features[col] == 1).sum()
        zeros = (df_features[col] == 0).sum()
        print(f"  {col}: 1={ones:,} ({ones/len(df_features)*100:.1f}%), 0={zeros:,} ({zeros/len(df_features)*100:.1f}%)")

# 새 피처 분포 확인
print("\n=== 새로 추가된 피처 분포 ===")
new_features = ['pulse_pressure', 'map_below_65', 'sofa_total', 'gender']
for col in new_features:
    if col in df_features.columns:
        print(f"  {col}: mean={df_features[col].mean():.2f}, std={df_features[col].std():.2f}")

# ICU binary 확인
icu_cols = [c for c in df_features.columns if c.startswith('icu_')]
if icu_cols:
    print(f"\n=== ICU Binary 피처 ({len(icu_cols)}개) ===")
    for col in icu_cols:
        ones = df_features[col].sum()
        print(f"  {col}: {ones:,} ({ones/len(df_features)*100:.1f}%)")


=== Flags 분포 ===
  lactate_missing: 1=99,940 (62.9%), 0=59,045 (37.1%)
  abga_checked: 1=61,459 (38.7%), 0=97,526 (61.3%)
  bilirubin_missing: 1=98,097 (61.7%), 0=60,888 (38.3%)
  urine_missing: 1=28,444 (17.9%), 0=130,541 (82.1%)

=== 새로 추가된 피처 분포 ===
  pulse_pressure: mean=52.40, std=16.17
  map_below_65: mean=0.34, std=0.47
  sofa_total: mean=5.92, std=3.25
  gender: mean=0.56, std=0.50

=== ICU Binary 피처 (5개) ===
  icu_micu: 76,648 (48.2%)
  icu_sicu: 89,327 (56.2%)
  icu_ccu: 11,399 (7.2%)
  icu_micu_sicu: 43,975 (27.7%)
  icu_tsicu: 12,905 (8.1%)


In [21]:
# 샘플 데이터 확인
print("\n=== 샘플 데이터 ===")
df_features.head()


=== 샘플 데이터 ===


,hr,rr,spo2,sbp,dbp,mbp,temp,hr_max,rr_max,spo2_min,...,urine_missing,hr_slope,sbp_slope,spo2_slope,rr_slope,temp_slope,lactate_slope,creatinine_slope,ph_slope,gcs_total_slope
0,47.0,17.5,94.0,92.0,34.0,56.0,36.800000,60.0,19.0,94.0,...,0,0.00,0.0,0.0,0.00,0.000000,0.0,0.0,0.000000e+00,0.0
1,47.0,17.5,94.0,120.0,34.0,56.0,37.055556,50.0,19.0,94.0,...,0,0.00,28.0,0.0,0.00,0.255556,0.0,0.0,2.842171e-14,0.0
2,47.5,18.0,94.0,117.0,45.0,74.0,37.055556,50.0,19.0,94.0,...,0,0.25,12.5,0.0,0.25,0.127778,0.0,0.0,0.000000e+00,0.0
3,47.5,17.5,94.0,117.0,43.0,68.0,37.055556,50.0,19.0,94.0,...,0,0.20,7.2,0.0,0.05,0.076667,0.0,0.0,5.684342e-15,0.0
4,46.5,17.0,96.0,117.0,43.0,68.0,37.055556,50.0,19.0,96.0,...,0,-0.05,4.7,0.4,-0.10,0.051111,0.0,0.0,0.000000e+00,0.0
